**Overview**

In this notebook, I build a machine learning model to predict the hourly bike rental count (cnt)
based on weather conditions and time-related features.
The dataset contains hourly records from a bike sharing system in London (2015–2016),
including temperature, humidity, wind speed, weather code, and whether the day is a holiday or weekend.


Model: LightGBM (Gradient Boosting)

Target: cnt — number of bikes rented per hour

Evaluation Metric: RMSE (Root Mean Squared Error)

In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

**1. Load Data**

We start by loading the training and test datasets.

train.csv — 13,931 rows with both features and the target (cnt)
test.csv — 3,484 rows with features only; we need to predict cnt for these rows

In [2]:
# Load Data
train = pd.read_csv('/kaggle/input/datasets/erenozeren/bike-sharing-prediction/train.csv')
test  = pd.read_csv('/kaggle/input/datasets/erenozeren/bike-sharing-prediction/test.csv')

**2. Feature Engineering**

The raw timestamp column alone is not useful for a model since it cannot interpret dates directly.
We extract meaningful time-based features from it and also create interaction features.

Cyclic encoding:
Hours and months are cyclical — hour 23 and hour 0 are neighbors, not opposites.
We encode them using sine and cosine transformations so the model understands this.

In [3]:
# Feature Engineering
def add_features(df):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
 
    # Time features
    df['hour']      = df['timestamp'].dt.hour       # 0-23
    df['dayofweek'] = df['timestamp'].dt.dayofweek  # 0=Mon, 6=Sun
    df['month']     = df['timestamp'].dt.month      # 1-12
    df['year']      = df['timestamp'].dt.year
    df['dayofyear'] = df['timestamp'].dt.dayofyear  # 1-365
 
    # Cyclic encoding (hour 23 and hour 0 are actually neighbors)
    df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
 
    # Interaction features
    df['temp_hum']   = df['t1'] * df['hum']   # temperature x humidity
    df['feels_diff'] = df['t1'] - df['t2']    # real temp - feels like temp
 
    return df
 
train = add_features(train)
test  = add_features(test)

**3. Define Features and Target**

We apply a log transformation to the target variable cnt.
The rental counts range from 0 to 7,860 — a wide spread that can bias the model toward large values.
log1p(cnt) compresses this range and leads to more stable training.
After prediction, we reverse this with expm1().

In [4]:
# Define Features and Target
FEATURES = [
    't1', 't2', 'hum', 'wind_speed', 'weather_code',
    'is_holiday', 'is_weekend', 'season',
    'hour', 'dayofweek', 'month', 'year', 'dayofyear',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'temp_hum', 'feels_diff'
]
 
X = train[FEATURES]
y = np.log1p(train['cnt'])   # log transform: balances the effect of large values

**4. Model — LightGBM**
   
We use LightGBM, a gradient boosting framework developed by Microsoft.
It builds decision trees sequentially — each tree corrects the mistakes of the previous one.
LightGBM is well-suited for this type of tabular data

In [5]:
# Define Model
model = LGBMRegressor(
    n_estimators     = 1500,  # 1500 decision trees built sequentially
    learning_rate    = 0.05,  # how strongly each tree learns
    num_leaves       = 127,   # complexity of the trees
    feature_fraction = 0.8,   # 80% of features used per tree
    bagging_fraction = 0.8,   # 80% of data used per tree
    bagging_freq     = 5,     # resample every 5 trees
    random_state     = 42,
    verbose          = -1
)

**5. Validation**
   
To evaluate the model before submitting, we split the training data by time:

First 85% → used to train the model
Last 15% → held out for validation (never seen by the model during training)

We use a time-based split (not random) because this is a time series dataset —
future data should not leak into training.

In [6]:
# Validation
split = int(len(X) * 0.85)  # last 15% held out, never shown to the model
 
model.fit(X.iloc[:split], y.iloc[:split])
 
val_pred  = model.predict(X.iloc[split:])
rmse_log  = np.sqrt(mean_squared_error(y.iloc[split:], val_pred))
rmse_orig = np.sqrt(mean_squared_error(
    np.expm1(y.iloc[split:]),
    np.expm1(val_pred)
))
 
print(f"Validation RMSE (log)   : {rmse_log:.4f}")
print(f"Validation RMSE (bikes) : {rmse_orig:.2f}")

Validation RMSE (log)   : 0.2164
Validation RMSE (bikes) : 271.36


**6. Retrain and Generate Submission**

We retrain the model on the full training data (including the validation split)
to make use of every available data point before predicting on the test set.

In [7]:
# Retain on Full Data + Generate Submission
model.fit(X, y)  # now includes the last 15% as well
 
preds = np.expm1(model.predict(test[FEATURES]))  # reverse the log transform
preds = np.clip(preds, 0, None)                  # ensure no negative values
 
sub = pd.DataFrame({'row_id': test['row_id'], 'cnt': preds})
sub.to_csv('submission.csv', index=False)
 
print("\nSubmission saved!")
print(sub['cnt'].describe().round(1))


Submission saved!
count    3483.0
mean     1095.0
std      1024.4
min        21.5
25%       268.6
50%       826.2
75%      1596.2
max      4963.5
Name: cnt, dtype: float64


**Results & Conclusion**
                        

Validation RMSE         ~271 bikes
 
Kaggle Public           268

The close match between validation and public score suggests the model generalizes well
and is not overfitting to the training data.